# Intro

## Global parameters

In [1]:
MAX_TIME_HOURS=1
ALPHA=0.01

## Modules

### Standard

In [2]:
import os, pickle, platform, sys
import numpy as np

In [3]:
from collections import defaultdict

In [4]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [5]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [6]:
from scipy.stats import spearmanr

In [7]:
from tqdm.notebook import tqdm, trange

In [8]:
import datetime as dt

In [9]:
from bowtie import edges2bowtie

### Home made

In [10]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [11]:
from auxiliary_functions import el2ks

In [12]:
from sam_bowtie import block_and_fluxes as bnf

## Load data

In [31]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'
BIPARTITE_FOLDER=HOME+'guarino_files/'

# Looking into the abyss

In [32]:
guarino_files=[file for file in os.listdir(BIPARTITE_FOLDER) if not file.startswith('.')]
guarino_files.sort()
guarino_files

['all_dico_labels.txt',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'crisi_dico_labels.pickle',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_sizes.csv',
 'ita_elections_dico_3_bowtie_sizes.csv',
 'ita_elections_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_5_bowtie_sizes.csv',
 'ita_elections_dico_6_bowtie_sizes.csv',
 'ita_elections_dico_labels.pickle',
 'quirinale_dico_0_bowtie_sizes.csv',
 'quirinale_dico_1_bowtie_sizes.csv',
 'quirinale_dico_2_bowtie_sizes.csv',
 'quirinale_dico_3_bowtie_sizes.csv',
 'quirinale_dico_4_bowtie_sizes.csv',
 'quirinale_dico_5_bowtie_sizes.csv',
 'quirinale_dico_6_bowtie_sizes.csv',
 'quirinale_dico_labels.pickle']

### Name of the various dicos

In [26]:
with open(BIPARTITE_FOLDER+guarino_files[0], 'r') as f:
    cacca=f.readlines()

In [27]:
cacca

['quirinale\n',
 '5: journalists & Media\n',
 '2: M5S\n',
 '0: journalists & IV & Azione & +Europa & Media\n',
 '4: Lega & FDI\n',
 '1: PD\n',
 '3: Media & journalists\n',
 '6: FI\n',
 '\n',
 'crisi\n',
 '1: Lega & FDI & FI\n',
 '2: M5S & journalists\n',
 '0: journalists & IV & Media & Azione & +Europa\n',
 '4: Media & journalists\n',
 '3: PD\n',
 '\n',
 'ita_elections\n',
 '1: PD & Media & +Europa & journalists\n',
 '2: M5S & Media\n',
 '3: journalists & IV & Azione\n',
 '0: Lega & FDI & FI\n',
 '4: journalists & Media (1)\n',
 '5: Media\n',
 '6: journalists & Media (2)\n',
 '\n']

In [34]:
with open(BIPARTITE_FOLDER+guarino_files[6], 'rb') as f:
    cacca = pickle.load(f)

In [35]:
cacca

{1: 'Lega & FDI & FI',
 2: 'M5S & journalists',
 0: 'journalists & IV & Media & Azione & +Europa',
 4: 'Media & journalists',
 3: 'PD'}

The pickle is what I need.

### From Guarino to my (actually, Sonnet's...) standards

In [52]:
cacca=np.genfromtxt(BIPARTITE_FOLDER+guarino_files[1], delimiter=',', dtype=int, skip_header=1)
header = np.genfromtxt(BIPARTITE_FOLDER+guarino_files[1], delimiter=',', dtype=str, max_rows=1)

In [53]:
block_dict={}
for i, name in enumerate(header):
    if name!='LSCC':
        block_dict[name] = cacca[:,i]
    else:
        block_dict['SCC'] = cacca[:,i]